# 22c · Recurrent PPO (sb3-contrib) on the initiation split (E22)

Experiment 22 asks whether another algorithm does better than experiment 16's Double DQN on the same task.
This notebook runs **Recurrent PPO (sb3-contrib)**; notebooks 22a, 22b and 22c run PPO, discrete SAC and Recurrent PPO the same way.

* **Algorithm:** sb3-contrib's `RecurrentPPO` with an LSTM policy, used unchanged behind `rltrade/agents/rppo.py`. Our batched GPU environment is wrapped as one Stable-Baselines3 VecEnv; episode ends are time limits and are bootstrapped. The LSTM sees the same 10 inputs as the other agents but keeps a memory of earlier steps.
* **Decision every 60 minutes** (E23a: with a decision every 5 minutes no agent setup learned signals as weak as
  the real factors; every 60 minutes Double DQN did). Otherwise the same as E16: state A (position, time in position,
  E15's 8 GRU factors), 0.10 % fee + 0.02 % slippage per side, orders filled at the next bar, reward = change in net
  asset value. E16's Double DQN decided every 5 minutes, so the comparison table marks it; E23c gives Double DQN at
  60 minutes on the same factors (with a different checkpoint protocol).
* **Tuning:** Optuna, 15 trials × 500k steps, scored by validation Sharpe (trial seeds 1000+).
* **Final runs:** 5 seeds × 2M steps with the tuned settings.
* **Checkpoint choice and fallback (as E16):** the best validation Sharpe among checkpoints with at least 5 trades;
  the run stays in cash unless that Sharpe beats both cash and buy and hold on validation.
* **Only validation is used.** Test and stress_bear stay zipped; they were used once, in experiment 17.

Outputs go to `thesis_rl/initiation/e22_algorithms/rppo/` and `initiation/report/e22_*`. The last section compares
every algorithm that has finished with E16's Double DQN.

Expected time on a T4: about 15-20 h. Every stage resumes after a disconnect (Optuna study and finished runs are on Drive),
so just run all again.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/14-initiation"
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
# sb3-contrib's RecurrentPPO (and stable-baselines3, gymnasium); exact versions are recorded in the environment report
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "stable-baselines3>=2.4,<3", "sb3-contrib>=2.4,<3"], check=True)
import sb3_contrib, stable_baselines3
print("stable-baselines3", stable_baselines3.__version__, "| sb3-contrib", sb3_contrib.__version__)

In [ ]:
seeding.record_environment(L.report / "environment", "22_rppo")

In [ ]:
import json, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt, optuna
from pathlib import Path
from rltrade.env import EnvConfig
from rltrade.runner import RunSpec, run, INVALID_SCORE
from rltrade import tune_rl, algo_compare
optuna.logging.set_verbosity(optuna.logging.WARNING)

INIT = L.root / "initiation"
AGENT, EXP_NAME = "rppo", "22_rppo"
OUT, REP = INIT / "e22_algorithms" / AGENT, INIT / "report"
for p in (OUT, REP): p.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/initiation_prepared") if "google.colab" in sys.modules else INIT / "prepared"
ZIP = INIT / "initiation_prepared.zip"
PREP = LOCAL / "fixed_plus"                         # only the market arrays (prices, signals) are read
NEEDED = [f"fixed_plus/market_{s}.npz" for s in ("train", "val")] + ["fixed_plus/meta.json"]
if not all((LOCAL / n).exists() for n in NEEDED):
    assert ZIP.exists(), f"upload initiation_prepared.zip to {ZIP}"
    with zipfile.ZipFile(ZIP) as z: z.extractall(LOCAL, members=NEEDED)   # test / stress_bear stay zipped
    print("unzipped", NEEDED, "->", LOCAL)

sel = json.loads((INIT / "selected_factors_e15.json").read_text())
F = Path(sel["factor_dir"])
if not F.exists():                                  # written on another machine: same place under initiation/
    parts = F.parts
    F = INIT.joinpath(*parts[parts.index("initiation") + 1:])
for s in ("train", "val"):
    f, m = np.load(F / f"factors_{s}.npy", mmap_mode="r"), np.load(PREP / f"market_{s}.npz")
    assert f.shape == (len(m["midpoint"]), 8), (s, f.shape)
    print(f"{s:5s} {f.shape[0]:7,d} bars | factors from {F.name} ({sel['choice']})")

TUNE_STATE, STATE, SEEDS = "A", "A", [0, 1, 2, 3, 4]   # the same as E16
STEP_GAP, TAG = 60, "g60"           # a decision every 60 bars (E23a); TAG marks the run ids
N_TRIALS, TUNE_STEPS = 15, 500_000
TOTAL_STEPS, EVAL_EVERY = 2_000_000, 100_000
MIN_TRADES = 5
FALLBACK = "cash"
EVAL = ("val",)                      # the only split scored in this notebook
AUTO_DISCONNECT = True               # release the Colab runtime when the last cell has written its outputs

## 1 · Baselines on validation (seconds)

In [ ]:
for b in ("buy_and_hold", "always_short", "cash"):
    run(RunSpec(EXP_NAME, b, env=EnvConfig(state="A", step_gap=STEP_GAP), tag=TAG), PREP, F, OUT, eval_splits=EVAL)

## 2 · Tune Recurrent PPO (sb3-contrib) on validation (state A, 15 trials × 500k steps)

In [ ]:
st = tune_rl.tune_agent(AGENT, PREP, F, OUT, env_cfg=EnvConfig(state=TUNE_STATE, step_gap=STEP_GAP), n_trials=N_TRIALS,
                        total_steps=TUNE_STEPS, eval_every=EVAL_EVERY, min_val_trades=MIN_TRADES, eval_splits=EVAL)
tr = st.trials_dataframe()
done = tr[tr.state == "COMPLETE"].copy()
show = ["number", "value", "user_attrs_val_sharpe", "user_attrs_val_cum_return_pct", "user_attrs_val_num_trades"] + \
       [c for c in done.columns if c.startswith("params_")]
print(f"{len(done)} complete, {(tr.state == 'PRUNED').sum()} pruned, "
      f"{(done.value > INVALID_SCORE).sum()} with an eligible checkpoint (>= {MIN_TRADES} validation trades)")
print(done.sort_values("value", ascending=False)[show].round(4).to_string(index=False))
if st.best_value > INVALID_SCORE:
    best_params = st.best_params
else:                                # no trial traded enough: take the best validation Sharpe and say so
    best_params = st.trials[int(done.sort_values("user_attrs_val_sharpe").number.iloc[-1])].params
    print("WARNING: no eligible trial - using the best validation Sharpe; expect stay-flat fallbacks")
AGENT_HP, ENV_OVER = tune_rl.params_to_hp(AGENT, best_params)
(OUT / "tuning" / "best_hp.json").write_text(json.dumps({"agent": AGENT, "params": best_params, "agent_hp": AGENT_HP,
    "env": ENV_OVER, "best_val_sharpe": st.best_value, "eligible": bool(st.best_value > INVALID_SCORE)},
    indent=2, default=list))
print("selected:", AGENT_HP, ENV_OVER)

## 3 · Final runs: 5 seeds × 2M steps

In [ ]:
hp = json.loads((OUT / "tuning" / "best_hp.json").read_text())
AGENT_HP, ENV_OVER = hp["agent_hp"], hp["env"]
AGENT_HP["net_dims"] = tuple(AGENT_HP["net_dims"])
bh_id = RunSpec(EXP_NAME, "buy_and_hold", env=EnvConfig(state="A", step_gap=STEP_GAP), tag=TAG).run_id
BH_VAL_SHARPE = json.loads((OUT / "models" / bh_id / "metrics.json").read_text())["val"]["sharpe"]
MIN_SCORE = max(0.0, BH_VAL_SHARPE)
print(f"a run must reach validation Sharpe > {MIN_SCORE:.3f} (cash 0, buy and hold {BH_VAL_SHARPE:+.3f}), else it stays flat")
for seed in SEEDS:
    run(RunSpec(EXP_NAME, AGENT, seed=seed, env=EnvConfig(state=STATE, step_gap=STEP_GAP, **ENV_OVER), agent_hp=AGENT_HP, tag=TAG,
                total_steps=TOTAL_STEPS, eval_every=EVAL_EVERY, min_val_trades=MIN_TRADES, fallback=FALLBACK,
                fallback_min_score=MIN_SCORE),
        PREP, F, OUT, eval_splits=EVAL)

## 4 · Results on validation, compared with experiment 16's Double DQN

In [ ]:
table, runs = algo_compare.compare(INIT)
mine = runs[runs.agent == AGENT].sort_values("seed")
per_run = mine[["seed", "best_step", "selection_eligible", "ckpt_val_sharpe", "ckpt_val_cum_return_pct",
                "ckpt_val_num_trades", "fallback_used", "val_sharpe", "val_cum_return_pct", "val_max_drawdown_pct",
                "val_num_trades", "minutes"]]
print("every Recurrent PPO (sb3-contrib) run (ckpt_* = the selected checkpoint; val_* = what is traded after the fallback rule)")
print(per_run.round(3).to_string(index=False))
print("\nvalidation, mean ± SD over seeds (every algorithm finished so far)\n" + table.to_string(index=False))
per_run.to_csv(REP / f"e22_{AGENT}_val_runs.csv", index=False)
table.to_csv(REP / "e22_algorithms_val.csv", index=False)
(REP / "e22_algorithms_val.md").write_text(
    "# E22 algorithm comparison, validation only\n\nEvery E22 algorithm decides every 60 minutes (E16 Double DQN: every 5); otherwise as E16 (state A, 15 Optuna trials "
    "x 500k steps, 5 seeds x 2M steps, >= 5 validation trades, stay-flat fallback).\n\n" + table.to_markdown(index=False)
    + "\n", encoding="utf-8")
(REP / f"e22_{AGENT}_val.md").write_text(f"# E22 Recurrent PPO (sb3-contrib), validation only\n\n" + per_run.round(3).to_markdown(index=False)
    + "\n", encoding="utf-8")

In [ ]:
fig = algo_compare.plot_equity(INIT, REP / "e22_val_equity.png"); plt.show()

In [ ]:
(INIT / f"selected_agent_e22_{AGENT}.json").write_text(json.dumps({
    "agent": AGENT, "agent_hp": AGENT_HP, "env": ENV_OVER, "state": STATE, "seeds": SEEDS, "factor_dir": str(F),
    "selection": f"best validation Sharpe among checkpoints with >= {MIN_TRADES} validation trades",
    "fallback": f"{FALLBACK} when no checkpoint qualifies or its validation Sharpe <= {MIN_SCORE:.4f}", "step_gap": STEP_GAP,
    "runs": sorted(mine.run_id), "models": str(OUT / "models"),
    "note": "test and stress_bear not used; they were evaluated once in experiment 17"}, indent=1, default=list))
print("wrote", REP / "e22_algorithms_val.md", "|", REP / "e22_val_equity.png")

In [ ]:
# release the GPU once everything above is on Drive (outputs stay visible in this notebook)
if AUTO_DISCONNECT and "google.colab" in sys.modules:
    from google.colab import runtime
    runtime.unassign()